# SmartJob Morocco
### Recherche intelligente hybride + filtres fiables + assistant RAG contrôlé

**Données ciblées :** les 1 542 offres ReKrute du projet, avec les champs `offer_id`, `title`, `company`, `location`, `publication_date`, `sector`, `contract_type`, `education_level`, `experience`, `remote`, `description` et `url`.

**Choix d'architecture :**
1. Préparer et structurer les annonces sans inventer les informations absentes.
2. Indexer le texte utile avec BM25 et `multilingual-e5-base`.
3. Fusionner les classements avec RRF.
4. Appliquer les filtres structurés sans les supprimer silencieusement.
5. Calculer les statistiques avec Pandas, jamais avec le LLM.
6. Activer Qwen 2.5-3B uniquement pour résumer/expliquer les annonces retrouvées, avec les sources.


## 0. Installation


In [8]:
%pip -q install pyarrow rank_bm25 sentence-transformers transformers accelerate

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration et imports


In [9]:
import os, re, json, math, hashlib, unicodedata, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

PARQUET_CANDIDATES = [
    "/kaggle/input/datasets/jalilajadouall/rekrute/rekrute_cleaned.parquet",
    "/kaggle/working/rekrute_cleaned.parquet",
    "rekrute_cleaned.parquet",
]
CSV_CANDIDATES = [
    "/kaggle/input/datasets/jalilajadouall/rekrute/rekrute_cleaned.csv",
    "/mnt/data/rekrute_raw.csv",
    "rekrute_cleaned.csv",
    "rekrute_raw.csv",
]
ARTIFACT_DIR = "/kaggle/working/smartjob_final_artifacts" if Path("/kaggle").exists() else "smartjob_final_artifacts"
Path(ARTIFACT_DIR).mkdir(parents=True, exist_ok=True)

EMBED_MODEL = "intfloat/multilingual-e5-base"
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
USE_LLM = True                 # mettre False pour tester seulement le moteur de recherche
CHUNK_CHARS = 750
TOP_K = 5
RRF_K = 60
RANDOM_STATE = 42

def first_existing(paths):
    return next((p for p in paths if os.path.exists(p)), None)

DATA_PATH = first_existing(PARQUET_CANDIDATES)
if DATA_PATH:
    df_raw = pd.read_parquet(DATA_PATH)
else:
    DATA_PATH = first_existing(CSV_CANDIDATES)
    if not DATA_PATH:
        raise FileNotFoundError(
            "Fichier introuvable. Ajoute rekrute_cleaned.parquet ou rekrute_raw.csv "
            "dans /kaggle/input, puis adapte les chemins dans PARQUET_CANDIDATES/CSV_CANDIDATES."
        )
    df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("Fichier :", DATA_PATH)
print("Dimensions :", df_raw.shape)
print("Colonnes :", df_raw.columns.tolist())
display(df_raw.head(3))

Fichier : /kaggle/input/datasets/jalilajadouall/rekrute/rekrute_cleaned.parquet
Dimensions : (1542, 14)
Colonnes : ['offer_id', 'source', 'title', 'company', 'location', 'publication_date', 'sector', 'contract_type', 'education_level', 'experience', 'remote', 'description', 'url', 'scraped_at']


,offer_id,source,title,company,location,publication_date,sector,contract_type,education_level,experience,remote,description,url,scraped_at
0,00ef79fff343f3ec,ReKrute,Compliance Manager H/F,Not Mentioned,Casablanca (Maroc,06/10/2026,Autres Industries,CDI,Bac +5 et plus minimum,Confirmé (5 à 10 ans),Non,Offres d'emploi\nIls ReKrutent\nConseils Karri...,https://www.rekrute.com/offre-emploi-complianc...,2026-11-07T12:22:37.837824+00:00
1,015a99371514f2da,ReKrute,Technicien de travaux - Immobilier (H/F),ALH Holding,Rabat (Maroc,13/10/2026,Not Mentioned,Autre,Bac +2 minimum,Intermédiaire (3 à 5 ans),Non,Offres d'emploi\nIls ReKrutent\nConseils Karri...,https://www.rekrute.com/offre-emploi-technicie...,2026-11-07T12:07:12.547608+00:00
2,01bba4915291d6dc,ReKrute,Responsable Groupe d'Agence (H/F),Fondation Arrawaj,National (Maroc,03/11/2026,Not Mentioned,CDI,Bac +4 minimum,Intermédiaire (3 à 5 ans),Non,Offres d'emploi\nIls ReKrutent\nConseils Karri...,https://www.rekrute.com/offre-emploi-responsab...,2026-11-07T11:16:05.279937+00:00


## 2. Validation du schéma
On harmonise les noms de colonnes, puis on vérifie les champs indispensables.

In [10]:
# Harmonisation tolérante des noms de colonnes
df_raw.columns = [str(c).strip().lower() for c in df_raw.columns]
ALIASES = {
    "id": "offer_id", "job_id": "offer_id", "git status": "offer_id", "poste": "title",
    "entreprise": "company", "ville": "location", "lieu": "location",
    "date": "publication_date", "description_poste": "description",
    "type_contrat": "contract_type", "niveau_etude": "education_level",
    "niveau_d_etude": "education_level", "teletravail": "remote",
}
df_raw = df_raw.rename(columns={k:v for k,v in ALIASES.items() if k in df_raw.columns and v not in df_raw.columns})

required = ["title", "description"]
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise ValueError(f"Colonnes indispensables absentes : {missing}. Colonnes reçues : {df_raw.columns.tolist()}")

for c in ["offer_id","company","location","publication_date","sector","contract_type",
          "education_level","experience","remote","url"]:
    if c not in df_raw.columns:
        df_raw[c] = None

df_raw["offer_id"] = df_raw["offer_id"].fillna(pd.Series(df_raw.index.astype(str), index=df_raw.index)).astype(str)
df_raw["title"] = df_raw["title"].fillna("").astype(str)
df_raw["description"] = df_raw["description"].fillna("").astype(str)
print("Offres chargées :", len(df_raw))
print("Identifiants uniques :", df_raw.offer_id.nunique())

Offres chargées : 1542
Identifiants uniques : 1542


## 3. Nettoyage prudent et extraction des sections
Le nettoyage retire surtout les espaces et le bruit de page

In [11]:
MISSING = {"", "nan", "none", "null", "not mentioned", "non renseigné", "non renseigne",
           "non précisé", "non precise", "n/a", "indifférent", "indifferent"}

def clean_text(x):
    if x is None or pd.isna(x): return ""
    x = unicodedata.normalize("NFKC", str(x))
    x = re.sub(r"https?://\S+", " ", x)
    x = re.sub(r"[ \t\xa0]+", " ", x)
    x = re.sub(r"\n{3,}", "\n\n", x)
    return x.strip()

def norm_value(x):
    x = clean_text(x)
    return None if x.lower() in MISSING else x

def norm_url(x):
    # Ne pas utiliser clean_text ici : cette fonction retire volontairement les URL.
    if x is None or pd.isna(x): return None
    value = str(x).strip()
    return None if value.lower() in MISSING else value

def city_value(x):
    x = norm_value(x)
    if not x: return None
    v = re.split(r"[(,;/|]", x)[0].strip()
    aliases = {"Fes":"Fès", "Meknes":"Meknès", "Kenitra":"Kénitra", "Sale":"Salé",
               "Tetouan":"Tétouan", "Casablanca-Settat":"Casablanca"}
    return aliases.get(v, v) or None

SECTION_RE = re.compile(
    r"^\s*(ENTREPRISE|CULTURE DE L['’]ENTREPRISE|POSTE|PROFIL RECHERCH[ÉE])\s*:?\s*$",
    re.I | re.M
)
END_MARKERS = ["Découvre si ce poste", "Partager cette offre", "LABELS REMPORTÉS", "Navigation rapide"]

def parse_sections(description):
    raw = clean_text(description)
    end_positions = [raw.find(m) for m in END_MARKERS if raw.find(m) >= 0]
    body = raw[:min(end_positions)] if end_positions else raw
    matches = list(SECTION_RE.finditer(body))
    result = {"poste":"", "profil":"", "entreprise_txt":""}
    if not matches:
        result["poste"] = body
        return result
    for i, m in enumerate(matches):
        key = m.group(1).upper()
        key = "profil" if key.startswith("PROFIL") else ("entreprise_txt" if key.startswith(("ENTREPRISE","CULTURE")) else "poste")
        stop = matches[i+1].start() if i+1 < len(matches) else len(body)
        result[key] = (result[key] + "\n" + body[m.end():stop]).strip()
    return result

def build_offer_table(raw):
    rows = []
    for r in raw.itertuples(index=False):
        d = r._asdict()
        desc = d.get("description", "") or ""
        sec = parse_sections(desc)
        rows.append({
            "offer_id": str(d.get("offer_id")),
            "title": clean_text(d.get("title")),
            "company": norm_value(d.get("company")),
            "city": city_value(d.get("location")),
            "publication_date": norm_value(d.get("publication_date")),
            "sector": norm_value(d.get("sector")),
            "contract": norm_value(d.get("contract_type")),
            "education": norm_value(d.get("education_level")),
            "experience": norm_value(d.get("experience")),
            "remote": norm_value(d.get("remote")),
            "url": norm_url(d.get("url")),
            "poste_txt": clean_text(sec["poste"]),
            "profil_txt": clean_text(sec["profil"]),
        })
    out = pd.DataFrame(rows).drop_duplicates("offer_id", keep="last").reset_index(drop=True)
    # Une seule graphie par ville ('CASABLANCA' / 'casablanca' / 'Casablanca' ; 'Fes' / 'Fès')
    _f = lambda s: "".join(ch for ch in unicodedata.normalize("NFKD", str(s).lower()) if not unicodedata.combining(ch))
    _spell = {}
    for _c in out.city.dropna():
        _spell.setdefault(_f(_c), Counter())[_c] += 1
    _canon = {k: max(v, key=lambda x: (x != x.upper() and x != x.lower(), v[x])) for k, v in _spell.items()}
    out["city"] = out.city.map(lambda c: _canon[_f(c)] if isinstance(c, str) else None)
    out["full_text"] = (out.title.fillna("") + "\n" + out.poste_txt.fillna("") + "\n" + out.profil_txt.fillna("")).str.strip()
    return out

offers = build_offer_table(df_raw)
print("Table préparée :", offers.shape)
print("Textes vides :", int((offers.full_text.str.len() < 40).sum()))
print("URLs conservées :", int(offers.url.notna().sum()), "/", len(offers))
print("Identifiants dupliqués après dédoublonnage :", int(offers.offer_id.duplicated().sum()))
display(offers[["title","company","city","contract","experience"]].head(8))

Table préparée : (1542, 14)
Textes vides : 5
URLs conservées : 1542 / 1542
Identifiants dupliqués après dédoublonnage : 0


,title,company,city,contract,experience
0,Compliance Manager H/F,None,Casablanca,CDI,Confirmé (5 à 10 ans)
1,Technicien de travaux - Immobilier (H/F),ALH Holding,Rabat,Autre,Intermédiaire (3 à 5 ans)
2,Responsable Groupe d'Agence (H/F),Fondation Arrawaj,National,CDI,Intermédiaire (3 à 5 ans)
3,Chef Departement Regional H/F,Groupe Label'Vie - Carrefour,Casablanca,CDI,Confirmé (5 à 10 ans)
4,Professeurs vacataires BTS,APEE - Association pour la Promotion de l’Ense...,Rabat,CDI,Confirmé (5 à 10 ans)
5,Consultant Expert Cryptographie & Chiffrement,Akkodis Maroc,Casablanca,CDI,Junior (1 à 3 ans)
6,Rejoignez nos équipes sinistres à casablanca !,Concentrix,Casablanca,Autre,Débutant (-1 an)
7,Directeur Regional H/F,Groupe Label'Vie - Carrefour,Casablanca,CDI,Expert (10 à 20 ans)


## 4. Rôles et technologies
La classification de rôle privilégie le titre. Les technologies sont extraites du titre et des sections de l'offre. Ce sont des règles explicables, pas un modèle entraîné.

In [12]:
ROLE_PATTERNS = [
    ("Data Scientist", r"data\s*scientist|machine learning engineer|ing[ée]nieur ia|ai engineer|nlp engineer|computer vision"),
    ("Data Engineer", r"data\s*engineer|ing[ée]nieur (de )?donn[ée]es|big\s*data engineer|data platform engineer|dataops"),
    ("Data Analyst", r"data\s*analyst|analyste (de )?donn[ée]es|analyste data"),
    ("BI / Décisionnel", r"business intelligence|\bbi analyst\b|d[ée]cisionnel|power\s*bi developer|consultant bi|d[ée]veloppeur (ms |power )?bi\b|ing[ée]nieur bi\b|chef de projet bi\b|data engineer bi\b"),
    ("Business Analyst", r"business analyst|analyste m[ée]tier|analyste fonctionnel"),
]
ROLE_RX = [(name, re.compile(pat, re.I)) for name,pat in ROLE_PATTERNS]

def classify_role(title):
    for name, rx in ROLE_RX:
        if rx.search(title or ""):
            return name
    return "Autre / non classé"

TECH_PATTERNS = {
    "Python":r"\bpython\b", "SQL":r"\bsql\b|t-sql|pl/sql", "Spark":r"\bspark\b|pyspark",
    "Power BI":r"power\s*-?bi", "Tableau":r"\btableau\b(?!\s+de\s+bord)", "Excel":r"\bexcel\b",
    "Azure":r"\bazure\b", "AWS":r"\baws\b|amazon web services", "GCP / BigQuery":r"\bgcp\b|google cloud|bigquery",
    "Snowflake":r"snowflake", "Databricks":r"databricks", "Docker":r"docker", "Kubernetes":r"kubernetes|\bk8s\b",
    "Airflow":r"airflow", "Kafka":r"kafka", "Hadoop":r"hadoop|hdfs|hive", "Talend":r"talend",
    "SSIS":r"\bssis\b", "dbt":r"\bdbt\b", "Java":r"\bjava\b(?!script)",
    "Scala":r"\bscala\b", "Git":r"\bgit(?:hub|lab)?\b", "PostgreSQL":r"postgres",
    "MongoDB":r"mongodb|\bmongo\b", "Oracle":r"oracle", "TensorFlow":r"tensorflow",
    "PyTorch":r"pytorch", "scikit-learn":r"scikit-learn|sklearn", "LLM / GenAI":r"\bllm\b|gen ?ai|ia g[ée]n[ée]rative|\brag\b",
    "ETL":r"\betl\b", "Data Warehouse":r"data warehouse|entrep[oô]t de donn[ée]es",
    "Linux":r"\blinux\b|\bunix\b", "DevOps / CI-CD":r"devops|ci/cd|jenkins",
}
TECH_RX = {k: re.compile(v, re.I) for k,v in TECH_PATTERNS.items()}

offers["role"] = offers.title.map(classify_role)
offers["techs"] = [
    [name for name,rx in TECH_RX.items() if rx.search((t or "") + "\n" + (p or "") + "\n" + (q or ""))]
    for t,p,q in zip(offers.title, offers.poste_txt, offers.profil_txt)
]
DATA_ROLES = ["Data Scientist","Data Engineer","Data Analyst","BI / Décisionnel","Business Analyst"]
data_offers = offers[offers.role.isin(DATA_ROLES)].copy()
print("Répartition des rôles :")
display(offers.role.value_counts().rename_axis("role").to_frame("offres"))
print("Offres classées dans les rôles Data/BI :", len(data_offers))
print("Détail des catégories Data/BI :")
display(data_offers.role.value_counts().rename_axis("role").to_frame("offres"))
print("Technologies les plus citées dans ces offres :")
display(pd.DataFrame(Counter(t for xs in data_offers.techs for t in xs).most_common(15), columns=["technologie","nombre_offres"]))

Répartition des rôles :


,offres
role,
Autre / non classé,1521
Data Engineer,6
Data Scientist,6
Data Analyst,4
BI / Décisionnel,3
Business Analyst,2


Offres classées dans les rôles Data/BI : 21
Détail des catégories Data/BI :


,offres
role,
Data Engineer,6
Data Scientist,6
Data Analyst,4
BI / Décisionnel,3
Business Analyst,2


Technologies les plus citées dans ces offres :


,technologie,nombre_offres
0,SQL,13
1,Power BI,11
2,Python,10
3,Git,8
4,DevOps / CI-CD,8
5,Spark,6
6,Databricks,6
7,Azure,5
8,ETL,3
9,Linux,3


## 5. Préparation des documents et chunks
Chaque annonce reste associée à son identifiant. On crée une fiche courte et des passages séparés POSTE / PROFIL pour limiter le mélange de contexte.

In [13]:
def offer_header(o):
    fields = [f"Poste : {o.title}"]
    for label,col in [("Entreprise","company"),("Ville","city"),("Contrat","contract"),
                      ("Expérience","experience"),("Formation","education")]:
        val = getattr(o,col)
        if val: fields.append(f"{label} : {val}")
    if o.techs: fields.append("Technologies détectées : " + ", ".join(o.techs))
    return " | ".join(fields)

def make_chunks(text, max_chars=CHUNK_CHARS):
    """Découpe les paragraphes en morceaux bornés, y compris les phrases très longues."""
    paras = [p.strip() for p in re.split(r"\n+", text or "") if p.strip()]
    out, cur = [], ""

    def split_long_piece(piece):
        # Découpe d'abord aux frontières de phrases, puis par caractères si nécessaire.
        sentences = re.split(r"(?<=[.!?;:])\s+", piece)
        pieces = []
        for sentence in sentences:
            sentence = sentence.strip()
            while len(sentence) > max_chars:
                pieces.append(sentence[:max_chars])
                sentence = sentence[max_chars:]
            if sentence:
                pieces.append(sentence)
        return pieces

    for para in paras:
        for piece in split_long_piece(para):
            if cur and len(cur) + len(piece) + 1 > max_chars:
                out.append(cur)
                cur = piece
            else:
                cur = (cur + "\n" + piece).strip()
    if cur:
        out.append(cur)
    return out

chunk_rows = []
for o in offers.itertuples(index=False):
    header = offer_header(o)
    # La fiche conserve les métadonnées ; les chunks textuels conservent les missions/profils.
    chunk_rows.append({"offer_id":o.offer_id, "section":"FICHE", "text":header})
    for section, value in [("POSTE",o.poste_txt),("PROFIL",o.profil_txt)]:
        for part in make_chunks(value):
            if len(part) >= 35:
                chunk_rows.append({"offer_id":o.offer_id, "section":section, "text":header + "\n" + part})
chunks = pd.DataFrame(chunk_rows)
print("Nombre d'offres :", len(offers), "| Nombre de chunks :", len(chunks))
display(chunks.head(3))

Nombre d'offres : 1542 | Nombre de chunks : 7060


,offer_id,section,text
0,00ef79fff343f3ec,FICHE,Poste : Compliance Manager H/F | Ville : Casab...
1,00ef79fff343f3ec,POSTE,Poste : Compliance Manager H/F | Ville : Casab...
2,00ef79fff343f3ec,POSTE,Poste : Compliance Manager H/F | Ville : Casab...


## 6. Recherche lexicale BM25
BM25 est utile pour retrouver précisément les termes techniques comme Spark, SSIS, Airflow ou Power BI.

In [14]:
# rank_bm25 est installé dans la cellule d'installation.
from rank_bm25 import BM25Okapi

def fold(text):
    text = unicodedata.normalize("NFKD", str(text or "").lower())
    return "".join(c for c in text if not unicodedata.combining(c))

def tokenize(text):
    return re.findall(r"[a-z0-9+#.]+", fold(text))

corpus_tokens = [tokenize(t) for t in chunks.text]
bm25 = BM25Okapi(corpus_tokens)
print("Index BM25 prêt :", len(corpus_tokens), "documents")

Index BM25 prêt : 7060 documents


## 7. Embeddings sémantiques
On utilise `multilingual-e5-base` avec les préfixes `query:` et `passage:` recommandés. Les vecteurs sont normalisés et mis en cache.

In [15]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBED_MODEL, device=device)
embedder.max_seq_length = 384
QUERY_PREFIX, PASSAGE_PREFIX = "query: ", "passage: "

cache_signature = hashlib.sha1(
    (EMBED_MODEL + "\n" + "\n".join(chunks.text.astype(str))).encode("utf-8")
).hexdigest()[:12]
EMB_PATH = f"{ARTIFACT_DIR}/embeddings_{cache_signature}.npy"

if os.path.exists(EMB_PATH):
    embeddings = np.load(EMB_PATH)
    print("Embeddings chargés depuis le cache.")
else:
    embeddings = embedder.encode(
        [PASSAGE_PREFIX + t for t in chunks.text],
        batch_size=48, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    np.save(EMB_PATH, embeddings)

print("Forme :", embeddings.shape, "| Device :", device)

def dense_scores(query):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype("float32")[0]
    return embeddings @ qv

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/148 [00:00<?, ?it/s]

Forme : (7060, 768) | Device : cuda


## 8. Filtres structurés stricts
Les filtres ne sont **jamais supprimés silencieusement**. Si aucun résultat ne satisfait tous les critères, le moteur le dit et propose de modifier la recherche. Pour plusieurs technologies, `ou` déclenche une logique OR ; sinon, la logique AND est utilisée.

In [16]:
CITY_VALUES = sorted([x for x in offers.city.dropna().unique()], key=len, reverse=True)
CONTRACT_VALUES = sorted([x for x in offers.contract.dropna().unique()], key=len, reverse=True)
ROLE_VALUES = DATA_ROLES

def parse_query_filters(query):
    q = fold(query)
    f = {}
    # Ville : correspondance sur les valeurs réellement présentes
    for city in CITY_VALUES:
        if fold(city) in q:
            f["city"] = city
            break
    # Rôle : termes du titre, sans classer sur une simple mention du corps
    role_terms = {
        "Data Scientist":r"data scientist|machine learning|nlp|computer vision",
        "Data Engineer":r"data engineer|ingenieur de donnees|big data engineer|data platform",
        "Data Analyst":r"data analyst|analyste de donnees|analyste data",
        "BI / Décisionnel":r"business intelligence|decisionnel|bi analyst",
        "Business Analyst":r"business analyst|analyste metier|analyste fonctionnel",
    }
    for role,pat in role_terms.items():
        if re.search(pat,q):
            f["role"] = role
            break
    for contract in CONTRACT_VALUES:
        if fold(contract) in q:
            f["contract"] = contract
            break
    detected = [tech for tech,rx in TECH_RX.items() if rx.search(query)]
    if detected:
        f["techs"] = detected
        f["tech_operator"] = "OR" if re.search(r"\bou\b",q) else "AND"
    if re.search(r"debutant|debutante|sans experience|premier emploi|junior",q):
        f["experience_level"] = "junior"
    if re.search(r"teletravail|a distance|remote|hybride",q):
        f["remote"] = True
    return f

def filter_offers(df, filters):
    mask = pd.Series(True,index=df.index)
    if "city" in filters: mask &= df.city.map(fold).eq(fold(filters["city"]))
    if "role" in filters: mask &= df.role.eq(filters["role"])
    if "contract" in filters: mask &= df.contract.map(fold).eq(fold(filters["contract"]))
    if "techs" in filters:
        op = filters.get("tech_operator","AND")
        if op == "OR":
            mask &= df.techs.map(lambda xs: any(t in xs for t in filters["techs"]))
        else:
            mask &= df.techs.map(lambda xs: all(t in xs for t in filters["techs"]))
    if filters.get("experience_level") == "junior":
        mask &= df.experience.fillna("").str.contains(r"(?i)junior|débutant|debutant|0.?1 an|sans expérience")
    if filters.get("remote"):
        mask &= df.remote.fillna("").str.contains(r"(?i)oui|hybride|remote|télétravail")
    return df[mask].copy()

test_q = "Data Engineer avec Spark ou Python à Casablanca"
print(parse_query_filters(test_q))
print("Nombre de correspondances :", len(filter_offers(offers,parse_query_filters(test_q))))

{'city': 'Casablanca', 'role': 'Data Engineer', 'techs': ['Python', 'Spark'], 'tech_operator': 'OR'}
Nombre de correspondances : 2


## 9. Recherche hybride et fusion RRF
Les scores BM25 et cosinus n'ont pas la même échelle : RRF fusionne les rangs. Les résultats sont dédoublonnés par annonce, pas par chunk.

In [17]:
def reciprocal_rank_fusion(rankings, k=RRF_K):
    scores = {}
    for ranking in rankings:
        for rank, item in enumerate(ranking, start=1):
            scores[item] = scores.get(item, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

chunk_offer_ids = chunks.offer_id.astype(str).tolist()
offer_by_id = {str(r.offer_id): r for r in offers.itertuples(index=False)}

def _rank_ids_by_max_score(scores, allowed_mask, min_score=None):
    """Classe les offres par leur meilleur score de chunk, sans favoriser le nombre de chunks."""
    best = {}
    for idx, score in enumerate(scores):
        if not allowed_mask[idx] or not np.isfinite(score):
            continue
        if min_score is not None and score <= min_score:
            continue
        oid = chunk_offer_ids[idx]
        best[oid] = max(best.get(oid, -np.inf), float(score))
    return sorted(best, key=best.get, reverse=True)

def retrieve(query, top_k=TOP_K, mode="hybrid", use_filters=True):
    if mode not in ("bm25", "dense", "hybrid"):
        raise ValueError("mode doit être 'bm25', 'dense' ou 'hybrid'.")

    filters = parse_query_filters(query) if use_filters else {}
    candidates = filter_offers(offers, filters) if filters else offers
    if candidates.empty:
        return [], {"filters": filters, "candidate_count": 0,
                    "message": "Aucune offre ne satisfait tous les filtres."}

    allowed = set(candidates.offer_id.astype(str))
    allowed_mask = np.array([oid in allowed for oid in chunk_offer_ids], dtype=bool)

    rankings = []
    chunk_rankings = []
    if mode in ("bm25", "hybrid"):
        bm25_scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=float)
        bm25_scores[~allowed_mask] = -np.inf
        chunk_order = [int(i) for i in np.argsort(-bm25_scores)
                       if np.isfinite(bm25_scores[i]) and bm25_scores[i] > 0]
        # Exclut les offres sans aucun terme lexical correspondant.
        offer_order = _rank_ids_by_max_score(bm25_scores, allowed_mask, min_score=0.0)
        rankings.append(offer_order)
        chunk_rankings.append(chunk_order)

    if mode in ("dense", "hybrid"):
        dense = np.asarray(dense_scores(query), dtype=float)
        dense[~allowed_mask] = -np.inf
        chunk_order = [int(i) for i in np.argsort(-dense)
                       if np.isfinite(dense[i])]
        offer_order = _rank_ids_by_max_score(dense, allowed_mask)
        rankings.append(offer_order)
        chunk_rankings.append(chunk_order)

    ranked_offers = rankings[0] if len(rankings) == 1 else reciprocal_rank_fusion(rankings)
    ranked_chunks = chunk_rankings[0] if len(chunk_rankings) == 1 else reciprocal_rank_fusion(chunk_rankings)

    # Associe à chaque offre ses passages les mieux classés, pas les deux premiers chunks arbitraires.
    chunks_by_offer = {}
    for idx in ranked_chunks:
        oid = chunk_offer_ids[idx]
        if oid not in chunks_by_offer:
            chunks_by_offer[oid] = []
        if len(chunks_by_offer[oid]) < 3:
            chunks_by_offer[oid].append(idx)

    hits = []
    for oid in ranked_offers:
        hits.append({"offer_id": oid, "chunk_ids": chunks_by_offer.get(oid, [])})
        if len(hits) >= top_k:
            break

    return hits, {"filters": filters, "candidate_count": len(candidates), "message": None}

def search(query, top_k=TOP_K, mode="hybrid", use_filters=True):
    hits,info = retrieve(query,top_k,mode,use_filters)
    print("Filtres :",info["filters"] or "aucun")
    print("Offres candidates après filtres :",info["candidate_count"])
    if not hits:
        print(info["message"] or "Aucun résultat.")
        return pd.DataFrame()
    rows=[]
    for rank,h in enumerate(hits,1):
        o=offer_by_id[h["offer_id"]]
        rows.append({
            "rang":rank,"poste":o.title,"entreprise":o.company or "Non précisée",
            "ville":o.city or "Non précisée","contrat":o.contract or "Non précisé",
            "expérience":o.experience or "Non précisée",
            "technologies détectées":", ".join(o.techs[:8]),
            "section retrouvée": (
                chunks.iloc[h["chunk_ids"][0]].section
                if h["chunk_ids"] else "Aucun passage retrouvé"
            ),
            "url":o.url
        })
    return pd.DataFrame(rows)

display(search("Data Engineer Spark à Casablanca",top_k=5))

Filtres : {'city': 'Casablanca', 'role': 'Data Engineer', 'techs': ['Spark'], 'tech_operator': 'AND'}
Offres candidates après filtres : 1


,rang,poste,entreprise,ville,contrat,expérience,technologies détectées,section retrouvée,url
0,1,Data Engineer (F/H),Sofrecom Maroc,Casablanca,CDI,Junior (1 à 3 ans),"Python, SQL, Spark, Airflow, dbt, Scala, Git",FICHE,https://www.rekrute.com/offre-emploi-data-engi...


## 10. Réponses analytiques exactes avec Pandas
Le LLM n'effectue ni comptage ni classement statistique. Les statistiques sont calculées directement sur les annonces filtrées.

In [18]:
ANALYTIC_RE = re.compile(
    r"combien|nombre d[' ]offres|how many|top\s*\d+|plus demand|répartition|repartition|"
    r"pourcentage|proportion|statistique|classement|quelles? (sont )?les (technologies|compétences|villes|entreprises|secteurs)",
    re.I
)

def is_analytic_question(q):
    return bool(ANALYTIC_RE.search(q))

def analytic_answer(query, topn=10):
    filters = parse_query_filters(query)
    sub = filter_offers(offers,filters) if filters else offers
    if sub.empty:
        return "Aucune offre ne correspond aux filtres détectés."
    q = fold(query)
    if re.search(r"technolog|competence|outil|stack",q):
        counts = Counter(t for xs in sub.techs for t in xs)
        return "\n".join([f"Technologies les plus citées sur {len(sub)} offres :"]+
                           [f"- {t} : {n} offres ({n/len(sub):.1%})" for t,n in counts.most_common(topn)])
    if re.search(r"ville|city",q):
        counts=sub.city.value_counts().head(topn)
        return "Répartition par ville :\n"+"\n".join(f"- {k}: {v}" for k,v in counts.items())
    if re.search(r"entreprise|société",q):
        counts=sub.company.value_counts().head(topn)
        return "Répartition par entreprise :\n"+"\n".join(f"- {k}: {v}" for k,v in counts.items())
    if re.search(r"secteur",q):
        counts=sub.sector.value_counts().head(topn)
        return "Répartition par secteur :\n"+"\n".join(f"- {k}: {v}" for k,v in counts.items())
    return f"{len(sub)} offre(s) correspondent aux filtres détectés, sur {len(offers)} annonces."

print(analytic_answer("Combien d'offres Data Analyst à Casablanca ?"))
print(analytic_answer("Quelles technologies sont les plus demandées pour Data Engineer ?"))

3 offre(s) correspondent aux filtres détectés, sur 1542 annonces.
Technologies les plus citées sur 6 offres :
- SQL : 6 offres (100.0%)
- Git : 6 offres (100.0%)
- Python : 5 offres (83.3%)
- DevOps / CI-CD : 5 offres (83.3%)
- Spark : 3 offres (50.0%)
- dbt : 3 offres (50.0%)
- Linux : 2 offres (33.3%)
- Kafka : 2 offres (33.3%)
- Scala : 2 offres (33.3%)
- Azure : 2 offres (33.3%)


## 11. Évaluation du moteur

La **P@5** correspond au nombre de résultats pertinents parmi les cinq premiers, divisé par 5. **Recall@5** mesure la proportion des offres pertinentes retrouvées dans les cinq premiers résultats. **MRR** mesure le rang du premier résultat pertinent.

L'évaluation automatique ci-dessous est un **contrôle de cohérence par règles**, pas un jugement humain indépendant : les offres dites pertinentes sont construites à partir des mêmes règles de rôle, ville et technologies que celles utilisées dans la préparation. Ces métriques sont indicatives (*proxy*) et ne constituent pas un benchmark indépendant.

La colonne **Récupération normalisée@5** est une métrique complémentaire définie ici comme :
`nombre de résultats pertinents retrouvés / min(5, nombre d'offres pertinentes dans le corpus)`.
Elle indique la part du maximum théoriquement récupérable dans les cinq premiers résultats. Ce n'est pas une P@5 standard.

Ne conclus pas que l'hybride est meilleur sans que les résultats recalculés le démontrent. Compare BM25, Dense et Hybrid sur les mêmes requêtes, puis complète par une évaluation humaine.


In [19]:
EVAL_CASES = [
    ("Data Analyst Casablanca", {"role":"Data Analyst","city":"Casablanca"}),
    ("Data Engineer Rabat", {"role":"Data Engineer","city":"Rabat"}),
    ("Data Scientist Python", {"role":"Data Scientist","techs":["Python"],"tech_operator":"AND"}),
    ("BI Power BI", {"role":"BI / Décisionnel","techs":["Power BI"],"tech_operator":"AND"}),
    ("Data Engineer Spark", {"role":"Data Engineer","techs":["Spark"],"tech_operator":"AND"}),
]

def evaluate_retriever(k=5):
    rows=[]
    for query,criteria in EVAL_CASES:
        relevant=set(filter_offers(offers,criteria).offer_id.astype(str))
        if not relevant: continue
        for mode in ["bm25","dense","hybrid"]:
            hits,_=retrieve(query,top_k=k,mode=mode,use_filters=False)
            got=[h["offer_id"] for h in hits]
            n_rel=sum(x in relevant for x in got)
            rows.append({"requête":query,"système":mode,"P@5":n_rel/k,"Récupération normalisée@5":n_rel/min(k,len(relevant)),
                         "Recall@5":n_rel/len(relevant),"MRR":next((1/(i+1) for i,x in enumerate(got) if x in relevant),0.0),
                         "pertinentes_corpus":len(relevant)})
    return pd.DataFrame(rows)

eval_results=evaluate_retriever()
display(eval_results.groupby("système")[["P@5","Récupération normalisée@5","Recall@5","MRR"]].mean().round(3))
display(eval_results)

,P@5,Récupération normalisée@5,Recall@5,MRR
système,,,,
bm25,0.6,0.883,0.883,0.85
dense,0.4,0.583,0.583,0.70
hybrid,0.6,0.883,0.883,0.85


,requête,système,P@5,Récupération normalisée@5,Recall@5,MRR,pertinentes_corpus
0,Data Analyst Casablanca,bm25,0.6,1.000000,1.000000,1.00,3
1,Data Analyst Casablanca,dense,0.4,0.666667,0.666667,1.00,3
2,Data Analyst Casablanca,hybrid,0.6,1.000000,1.000000,1.00,3
3,Data Engineer Rabat,bm25,0.8,1.000000,1.000000,1.00,4
4,Data Engineer Rabat,dense,0.6,0.750000,0.750000,1.00,4
5,Data Engineer Rabat,hybrid,0.8,1.000000,1.000000,1.00,4
6,Data Scientist Python,bm25,0.6,0.750000,0.750000,1.00,4
7,Data Scientist Python,dense,0.4,0.500000,0.500000,0.25,4
8,Data Scientist Python,hybrid,0.6,0.750000,0.750000,1.00,4
9,BI Power BI,bm25,0.4,0.666667,0.666667,0.25,3


## 12. Validation manuelle

Le fichier `relevance_to_label.csv` contient les cinq premiers résultats de huit requêtes réalistes (40 lignes si chaque requête retourne cinq résultats).

Attribue un label à chaque ligne :
- `2` : très pertinent ;
- `1` : partiellement pertinent ;
- `0` : non pertinent.

Cette validation est distincte de l'évaluation automatique. Tant que les labels ne sont pas remplis et analysés, ne présente pas de score humain de pertinence.


In [20]:
GOLD_QUERIES = [
    "analyser des données et construire des tableaux de bord Power BI",
    "stage en data science ou intelligence artificielle",
    "data engineer avec Spark et Airflow à Casablanca",
    "ingénieur cloud avec Kubernetes et Docker",
    "je débute et je cherche un poste d'analyste de données",
    "responsable gouvernance et qualité des données",
    "business analyst dans la banque ou l'assurance",
    "ingénieur machine learning NLP",
]

def make_manual_judgement_template(path=f"{ARTIFACT_DIR}/relevance_to_label.csv"):
    rows=[]
    for qid,q in enumerate(GOLD_QUERIES,1):
        hits,_=retrieve(q,top_k=5,mode="hybrid",use_filters=False)
        for rank,h in enumerate(hits,1):
            o=offer_by_id[h["offer_id"]]
            best_text=" ".join(chunks.iloc[i].text for i in h["chunk_ids"][:2])
            rows.append({"query_id":qid,"query":q,"rank":rank,"offer_id":h["offer_id"],
                         "title":o.title,"company":o.company,"city":o.city,
                         "excerpt":best_text[:350],"label":""})
    out=pd.DataFrame(rows)
    out.to_csv(path,index=False,encoding="utf-8-sig")
    print(f"Fichier créé : {path} | {len(out)} lignes à noter.")
    return out

manual_template=make_manual_judgement_template()
display(manual_template.head(10))

Fichier créé : /kaggle/working/smartjob_final_artifacts/relevance_to_label.csv | 40 lignes à noter.


,query_id,query,rank,offer_id,title,company,city,excerpt,label
0,1,analyser des données et construire des tableau...,1,1486040d35eb7a05,Power BI Data Analyst (Stagiaire),Coface,Casablanca,Poste : Power BI Data Analyst (Stagiaire) | En...,
1,1,analyser des données et construire des tableau...,2,eff19d8a6374579e,Stagiaire en Business Intelligence & Power BI H/F,Sothema,Casablanca,Poste : Stagiaire en Business Intelligence & P...,
2,1,analyser des données et construire des tableau...,3,894ad1e9f9992825,Data Analyst (Finance),CAC (Centrale Automobile Chérifienne),Casablanca,Poste : Data Analyst (Finance) | Entreprise : ...,
3,1,analyser des données et construire des tableau...,4,a00a9fb0bb763609,Chargé de Projets SI (H/F),CRIT,Casablanca,Poste : Chargé de Projets SI (H/F) | Entrepris...,
4,1,analyser des données et construire des tableau...,5,46c6d73b35c0abab,Data Engineer Confirmé,Orange Business,Rabat,Poste : Data Engineer Confirmé | Entreprise : ...,
5,2,stage en data science ou intelligence artifici...,1,1486040d35eb7a05,Power BI Data Analyst (Stagiaire),Coface,Casablanca,Poste : Power BI Data Analyst (Stagiaire) | En...,
6,2,stage en data science ou intelligence artifici...,2,f4d766407dc618e9,Data Scientist / Ingénieur IA confirmé,CAC (Centrale Automobile Chérifienne),Casablanca,Poste : Data Scientist / Ingénieur IA confirmé...,
7,2,stage en data science ou intelligence artifici...,3,8405faa3cd871cb2,Spécialiste en Intelligence Artificielle / Cas...,O&D Services,Casablanca,Poste : Spécialiste en Intelligence Artificiel...,
8,2,stage en data science ou intelligence artifici...,4,eff19d8a6374579e,Stagiaire en Business Intelligence & Power BI H/F,Sothema,Casablanca,Poste : Stagiaire en Business Intelligence & P...,
9,2,stage en data science ou intelligence artifici...,5,e1653713f2e1d2fa,Data scientist - OTA (Databricks / PySpark),ALTEN Maroc,Fès,Poste : Data scientist - OTA (Databricks / PyS...,


## 13. Assistant RAG avec Qwen — optionnel
Le LLM est chargé seulement maintenant. Les sources sont construites à partir des annonces récupérées. Les métadonnées structurées priment sur le texte généré pour les conditions.

In [21]:
if USE_LLM:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer=AutoTokenizer.from_pretrained(LLM_MODEL)
    load_kwargs={"device_map":"auto"}
    if torch.cuda.is_available():
        load_kwargs["torch_dtype"]=torch.float16
    llm=AutoModelForCausalLM.from_pretrained(LLM_MODEL,**load_kwargs)
    llm.eval()
    print("LLM chargé :",LLM_MODEL,"| device :",llm.device)
else:
    tokenizer=None
    llm=None
    print("USE_LLM=False : moteur de recherche seul.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM chargé : Qwen/Qwen2.5-3B-Instruct | device : cuda:0


## 14. Génération contrôlée, citations et résumés
La vérification des citations contrôle que les numéros cités existent. Elle ne garantit pas à elle seule que chaque phrase est sémantiquement justifiée : les sources doivent toujours être vérifiées.

In [22]:
SYSTEM_PROMPT = (
    "Tu es SmartJob Morocco. Réponds en français uniquement à partir des extraits fournis. "
    "N'invente aucune compétence, expérience, salaire, date ou condition. "
    "Chaque information factuelle doit être suivie d'une citation [n]. "
    "Si les sources ne permettent pas de répondre, dis exactement : "
    "'Cette information n'apparaît pas dans les offres retrouvées.' "
    "Les champs structurés (expérience, contrat, ville, formation) sont des données de référence. "
    "Si le texte libre contredit un champ structuré, signale explicitement la contradiction "
    "et ne transforme jamais une durée du texte libre en exigence certaine. "
    "Distingue les annonces entre elles et signale toute contradiction au lieu de la résoudre. "
    "Ne déduis jamais une expérience, un diplôme ou une condition à partir du seul intitulé du poste. "
    "Pour toute question portant uniquement sur les champs structurés, reprends leur valeur exacte."
)

def build_rag_context(hits, max_chars_per_offer=2400):
    blocks=[]
    for n,h in enumerate(hits,1):
        o=offer_by_id[h["offer_id"]]
        parts=["MÉTADONNÉES STRUCTURÉES (à privilégier et à ne pas contredire) :",
               f"Poste : {o.title}",f"Entreprise : {o.company or 'non précisée'}",
               f"Ville : {o.city or 'non précisée'}",f"Contrat : {o.contract or 'non précisé'}",
               f"Expérience structurée : {o.experience or 'non précisée'}",
               f"Formation : {o.education or 'non précisée'}"]
        if o.techs: parts.append("Technologies détectées par règles : "+", ".join(o.techs))
        # Inclure les passages récupérés, sans concaténer les fiches des autres offres.
        passages=[]
        for idx in h["chunk_ids"]:
            row=chunks.iloc[idx]
            if row.section!="FICHE":
                passages.append(f"[{row.section}] {row.text}")
        text="\n".join(parts+passages)
        blocks.append(f"[{n}]\n{text[:max_chars_per_offer]}")
    return "\n\n".join(blocks)

@torch.inference_mode() if USE_LLM else (lambda f:f)
def generate_rag_answer(question,hits,max_new_tokens=320):
    if not USE_LLM:
        raise RuntimeError("Active USE_LLM=True puis exécute les cellules 13 et 14.")
    context=build_rag_context(hits)
    messages=[
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":f"SOURCES :\n{context}\n\nQUESTION : {question}\nRéponds brièvement avec les citations :"}
    ]
    prompt=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tokenizer(prompt,return_tensors="pt").to(llm.device)
    output=llm.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,
                        repetition_penalty=1.0,pad_token_id=tokenizer.eos_token_id)
    answer=tokenizer.decode(output[0][inputs.input_ids.shape[1]:],skip_special_tokens=True).strip()
    # Certains décodages peuvent ajouter plusieurs virgules après une citation.
    # On les retire sans modifier les virgules normales à l'intérieur des phrases.
    answer=re.sub(r"(\[\d+\])(?:\s*,\s*)+", r"\1", answer)
    answer=re.sub(r",\s*,+", ",", answer)
    cited=sorted({int(x) for x in re.findall(r"\[(\d+)\]",answer)})
    invalid=[x for x in cited if x<1 or x>len(hits)]
    for x in invalid:
        answer=answer.replace(f"[{x}]","")
    valid_cited = [x for x in cited if 1 <= x <= len(hits)]
    if not valid_cited and "n'apparait pas" not in fold(answer):
        answer += "\n\n[À vérifier : aucune citation valide détectée.]"
    return answer

def print_sources(hits):
    print("\nSources :")
    for n,h in enumerate(hits,1):
        o=offer_by_id[h["offer_id"]]
        print(f"[{n}] {o.title} | {o.company or 'Non précisée'} | {o.city or 'Non précisée'} | {o.url or 'URL absente'}")

def ask(question,top_k=TOP_K):
    if is_analytic_question(question):
        answer=analytic_answer(question)
        print("Réponse calculée avec Pandas :\n"+answer)
        return answer
    hits,info=retrieve(question,top_k=top_k,mode="hybrid",use_filters=True)
    print("Filtres appliqués :",info["filters"] or "aucun")
    print("Offres candidates :",info["candidate_count"])
    if not hits:
        print("Aucune annonce ne satisfait les filtres. Modifie ou retire un critère.")
        return None
    if not USE_LLM:
        display(search(question,top_k=top_k))
        print_sources(hits)
        return hits
    answer=generate_rag_answer(question,hits)
    print(answer)
    print_sources(hits)
    return answer

In [23]:
import re

# Test du nettoyage des citations
sample = "Réponse fondée sur la source [1],,,,,,"

cleaned = re.sub(r"(\[\d+\])(?:\s*,\s*)+", r"\1", sample)
cleaned = re.sub(r",\s*,+", ",", cleaned)
cleaned = re.sub(r"\s+([,.;:])", r"\1", cleaned)
cleaned = re.sub(r"[,;:]+\s*$", "", cleaned).strip()

print("Résultat :", cleaned)

assert cleaned == "Réponse fondée sur la source [1]"

print("Test de nettoyage des citations : OK")

Résultat : Réponse fondée sur la source [1]
Test de nettoyage des citations : OK


## 15. Démonstrations
Ces requêtes couvrent recherche exacte, langage naturel, statistiques, stages et refus d'inventer les salaires.

In [24]:
display(search("Data Engineer Spark à Casablanca",top_k=5))

Filtres : {'city': 'Casablanca', 'role': 'Data Engineer', 'techs': ['Spark'], 'tech_operator': 'AND'}
Offres candidates après filtres : 1


,rang,poste,entreprise,ville,contrat,expérience,technologies détectées,section retrouvée,url
0,1,Data Engineer (F/H),Sofrecom Maroc,Casablanca,CDI,Junior (1 à 3 ans),"Python, SQL, Spark, Airflow, dbt, Scala, Git",FICHE,https://www.rekrute.com/offre-emploi-data-engi...


In [25]:
ask("Quel est le profil recherché pour un Data Engineer avec Spark à Casablanca ?")

Filtres appliqués : {'city': 'Casablanca', 'role': 'Data Engineer', 'techs': ['Spark'], 'tech_operator': 'AND'}
Offres candidates : 1
Le profil recherché pour un Data Engineer avec Spark à Casablanca est un Junior (1 à 3 ans) possédant un Bac +5 et plus minimum. Il doit avoir des connaissances en Python, SQL, Spark, Airflow, dbt, Scala et Git. De plus, il doit garantir la qualité des livrables logiciels et documentaires, dans le respect des délais et des charges, et accompagner les développeurs juniors pour aider à l’avancement du projet. [1]

Sources :
[1] Data Engineer (F/H) | Sofrecom Maroc | Casablanca | https://www.rekrute.com/offre-emploi-data-engineer-fh-recrutement-sofrecom-maroc-casablanca-188166.html


'Le profil recherché pour un Data Engineer avec Spark à Casablanca est un Junior (1 à 3 ans) possédant un Bac +5 et plus minimum. Il doit avoir des connaissances en Python, SQL, Spark, Airflow, dbt, Scala et Git. De plus, il doit garantir la qualité des livrables logiciels et documentaires, dans le respect des délais et des charges, et accompagner les développeurs juniors pour aider à l’avancement du projet. [1]'

In [26]:
ask("Combien d'offres Data Analyst à Casablanca ?")

Réponse calculée avec Pandas :
3 offre(s) correspondent aux filtres détectés, sur 1542 annonces.


'3 offre(s) correspondent aux filtres détectés, sur 1542 annonces.'

In [27]:
ask("Y a-t-il des stages Power BI pour débutant ?")

Filtres appliqués : {'contract': 'Stage', 'techs': ['Power BI'], 'tech_operator': 'AND', 'experience_level': 'junior'}
Offres candidates : 2
Cette information n'apparaît pas dans les offres retrouvées.

Sources :
[1] Power BI Data Analyst (Stagiaire) | Coface | Casablanca | https://www.rekrute.com/offre-emploi-power-bi-data-analyst-stagiaire-recrutement-coface-casablanca-187698.html
[2] Stagiaire en Business Intelligence & Power BI H/F | Sothema | Casablanca | https://www.rekrute.com/offre-emploi-stagiaire-en-business-intelligence-power-bi-hf-recrutement-sothema-casablanca-188104.html


"Cette information n'apparaît pas dans les offres retrouvées."

In [28]:
ask("Quel salaire propose une offre Data Engineer à Rabat ?")

Filtres appliqués : {'city': 'Rabat', 'role': 'Data Engineer'}
Offres candidates : 4
Cette information n'apparaît pas dans les offres retrouvées.

Sources :
[1] Data Engineer Java / Scala Expérimenté – Data Platform | ALTEN Maroc | Rabat | https://www.rekrute.com/offre-emploi-data-engineer-java-scala-experimente-data-platform-recrutement-alten-maroc-rabat-187949.html
[2] Software Integration, API & Data Engineer(F/H) | Sofrecom Maroc | Rabat | https://www.rekrute.com/offre-emploi-software-integration-api-data-engineerfh-recrutement-sofrecom-maroc-rabatcasa-187556.html
[3] Data Engineer Confirmé | Orange Business | Rabat | https://www.rekrute.com/offre-emploi-data-engineer-confirme-recrutement-orange-business-rabatcasablanca-187669.html
[4] Senior Data Engineer | AXA GBS Morocco | Rabat | https://www.rekrute.com/offre-emploi-senior-data-engineer-recrutement-axa-gbs-morocco-rabat-180969.html


"Cette information n'apparaît pas dans les offres retrouvées."

## 16. Sauvegarde des artefacts
On sauvegarde les tables préparées, les chunks, les embeddings et la configuration. Le modèle LLM n'est pas recopié dans le dossier.

In [29]:
offers_export=offers.copy()
offers_export["techs"]=offers_export.techs.map(lambda xs: json.dumps(xs,ensure_ascii=False))
offers_export.to_parquet(f"{ARTIFACT_DIR}/offers_clean.parquet",index=False)
chunks.to_parquet(f"{ARTIFACT_DIR}/chunks.parquet",index=False)
np.save(f"{ARTIFACT_DIR}/embeddings.npy",embeddings)
config={
    "project":"SmartJob Morocco",
    "source":str(DATA_PATH),
    "n_offers":int(len(offers)),
    "n_chunks":int(len(chunks)),
    "embedding_model":EMBED_MODEL,
    "llm_model":LLM_MODEL if USE_LLM else None,
    "chunk_chars":CHUNK_CHARS,
    "retrieval":"BM25 + multilingual-e5 + RRF",
    "filters":"strict; no silent relaxation",
    "created_at":pd.Timestamp.utcnow().isoformat()
}
with open(f"{ARTIFACT_DIR}/config.json","w",encoding="utf-8") as f:
    json.dump(config,f,ensure_ascii=False,indent=2)
print("Artefacts sauvegardés :",sorted(os.listdir(ARTIFACT_DIR)))

Artefacts sauvegardés : ['chunks.parquet', 'config.json', 'embeddings.npy', 'embeddings_0b921decc1be.npy', 'offers_clean.parquet', 'relevance_to_label.csv']


## 18. Export de l'application Streamlit

Après avoir exécuté toutes les cellules précédentes, cette cellule crée une archive contenant `app.py`, les dépendances, le guide de lancement et les artefacts recalculés (`offers_clean.parquet`, `chunks.parquet`, `embeddings.npy`, `config.json`). Télécharge ensuite `smartjob_streamlit_app_bundle.zip` depuis l'espace de sortie Kaggle.

In [30]:
# Export de l'application Streamlit et des artefacts recalculés
from pathlib import Path
import zipfile

APP_SOURCE = 'from __future__ import annotations\n\nimport json\nimport re\nimport unicodedata\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\nfrom rank_bm25 import BM25Okapi\n\n# ---------------------------------------------------------------------\n# Configuration\n# ---------------------------------------------------------------------\nst.set_page_config(\n    page_title="SmartJob Morocco",\n    page_icon="💼",\n    layout="wide",\n    initial_sidebar_state="expanded",\n)\n\nBASE_DIR = Path(__file__).resolve().parent\nARTIFACT_DIR = Path(__import__("os").environ.get("SMARTJOB_ARTIFACT_DIR", BASE_DIR / "smartjob_final_artifacts"))\nOFFERS_FILE = ARTIFACT_DIR / "offers_clean.parquet"\nCHUNKS_FILE = ARTIFACT_DIR / "chunks.parquet"\nEMBEDDINGS_FILE = ARTIFACT_DIR / "embeddings.npy"\nCONFIG_FILE = ARTIFACT_DIR / "config.json"\n\nst.markdown("""\n<style>\n:root{--page:#EAF1F7;--panel:#fff;--navy:#174F7A;--blue:#3560A8;--green:#205B50;--text:#252B33;--muted:#687481;--line:#D7E0E9}\n.stApp{background:var(--page);color:var(--text)} .block-container{max-width:1420px;padding:1.25rem 1.6rem 2.4rem}\nheader[data-testid="stHeader"]{background:transparent}\nh1,h2,h3{color:var(--navy);letter-spacing:-.02em} h1{font-size:1.75rem!important;font-weight:750!important;margin-bottom:.1rem}\nh2{font-size:1.12rem!important;font-weight:700!important} h3{font-size:1rem!important;font-weight:650!important}\np,label,.stMarkdown{color:var(--text)} [data-testid="stSidebar"]{background:#F7FAFD;border-right:1px solid var(--line)}\n[data-testid="stSidebar"] h2{font-size:1.05rem!important}\n.main-title{padding:.25rem 0 .85rem;border-bottom:1px solid var(--line);margin-bottom:1rem}\n.main-title .eyebrow{font-size:.76rem;font-weight:700;color:var(--blue);text-transform:uppercase;letter-spacing:.09em}\n[data-testid="stMetric"]{background:#fff;border:1px solid var(--line);border-radius:8px;padding:.8rem 1rem;box-shadow:0 1px 2px rgba(25,45,70,.035)}\n[data-testid="stMetricLabel"]{color:var(--muted);font-size:.82rem}\n[data-testid="stMetricValue"]{color:var(--green);font-size:1.85rem;font-weight:750}\n.stTextInput input,.stSelectbox [data-baseweb="select"]>div{background:#fff;border:1px solid #C6D2DE;border-radius:5px}\ndiv.stButton>button,div.stFormSubmitButton>button{border-radius:5px;min-height:2.55rem;font-weight:650}\ndiv.stButton>button[kind="primary"],div.stFormSubmitButton>button[kind="primary"]{background:var(--blue);border:1px solid var(--blue);color:white}\ndiv.stButton>button[kind="primary"]:hover,div.stFormSubmitButton>button[kind="primary"]:hover{background:var(--navy);border-color:var(--navy)}\n[data-testid="stAlert"]{border-radius:5px}\n.job-card{background:#fff;border:1px solid var(--line);border-left:4px solid var(--blue);border-radius:6px;padding:1rem 1.1rem;margin:.55rem 0 .85rem}\n.job-meta{color:var(--muted);font-size:.88rem;margin:.2rem 0 .55rem}\n.pill{display:inline-block;padding:.15rem .48rem;margin:.1rem .22rem .1rem 0;border:1px solid #D5E0EB;border-radius:4px;background:#F1F5F9;color:#315477;font-size:.76rem}\na{color:var(--blue)!important;font-weight:650} hr{border-color:var(--line)}\n.section-label{font-size:.76rem;color:var(--muted);text-transform:uppercase;letter-spacing:.07em;font-weight:700}\n</style>\n""", unsafe_allow_html=True)\n\n\ndef fold(value: object) -> str:\n    value = unicodedata.normalize("NFKD", str(value or "").lower())\n    return "".join(ch for ch in value if not unicodedata.combining(ch))\n\n\ndef tokenize(value: str) -> list[str]:\n    return re.findall(r"[a-z0-9+#.]+", fold(value))\n\n\n@st.cache_resource(show_spinner="Chargement de l\'index de recherche…")\ndef load_data():\n    missing = [p.name for p in [OFFERS_FILE, CHUNKS_FILE, EMBEDDINGS_FILE] if not p.exists()]\n    if missing:\n        raise FileNotFoundError(\n            "Artefacts manquants : " + ", ".join(missing) +\n            ". Exécute d\'abord le notebook dans Kaggle, télécharge le dossier "\n            "\'smartjob_final_artifacts\', puis place-le à côté de app.py."\n        )\n    offers = pd.read_parquet(OFFERS_FILE)\n    chunks = pd.read_parquet(CHUNKS_FILE)\n    embeddings = np.load(EMBEDDINGS_FILE).astype("float32")\n    with open(CONFIG_FILE, encoding="utf-8") as f:\n        config = json.load(f)\n    if len(chunks) != len(embeddings):\n        raise ValueError(\n            f"Incohérence artefacts : {len(chunks)} chunks mais {len(embeddings)} embeddings."\n        )\n    offers["offer_id"] = offers["offer_id"].astype(str)\n    chunks["offer_id"] = chunks["offer_id"].astype(str)\n    offers["techs"] = offers["techs"].apply(\n        lambda x: json.loads(x) if isinstance(x, str) else (x if isinstance(x, list) else [])\n    )\n    corpus_tokens = [tokenize(t) for t in chunks["text"].fillna("")]\n    bm25_index = BM25Okapi(corpus_tokens)\n    return offers, chunks, embeddings, config, bm25_index\n\n\n@st.cache_resource(show_spinner="Chargement du moteur sémantique…")\ndef load_embedder(model_name: str):\n    try:\n        from sentence_transformers import SentenceTransformer\n        return SentenceTransformer(model_name, device="cpu")\n    except Exception as exc:\n        raise RuntimeError(\n            "Le moteur sémantique n\'a pas pu être chargé. "\n            "Sous Windows avec Python 3.8, installe les versions compatibles "\n            "du fichier requirements.txt. Pour l\'erreur protobuf, exécute : "\n            "python -m pip install --force-reinstall protobuf==3.20.3 "\n            "puis redémarre Streamlit. Détail technique : " + str(exc)\n        ) from exc\n\n\n@st.cache_resource(show_spinner="Chargement de Qwen — cette opération peut être longue…")\ndef load_qwen(model_name: str):\n    import torch\n    from transformers import AutoTokenizer, AutoModelForCausalLM\n    tokenizer = AutoTokenizer.from_pretrained(model_name)\n    model = AutoModelForCausalLM.from_pretrained(\n        model_name,\n        torch_dtype=torch.float32,\n        low_cpu_mem_usage=True\n    )\n    model.to("cpu")\n    model.eval()\n    return tokenizer, model\n\n\ndef generate_answer(question: str, results, chunks, tokenizer, model):\n    import torch\n    source_blocks = []\n    for n, (offer, chunk_ids) in enumerate(results, start=1):\n        meta = [\n            f"Poste : {getattr(offer, \'title\', \'\')}",\n            f"Entreprise : {getattr(offer, \'company\', None) or \'non précisée\'}",\n            f"Ville : {getattr(offer, \'city\', None) or \'non précisée\'}",\n            f"Contrat : {getattr(offer, \'contract\', None) or \'non précisé\'}",\n            f"Expérience structurée : {getattr(offer, \'experience\', None) or \'non précisée\'}",\n            f"Formation : {getattr(offer, \'education\', None) or \'non précisée\'}",\n        ]\n        passages = []\n        for idx in chunk_ids[:2]:\n            if 0 <= idx < len(chunks):\n                passages.append(str(chunks.iloc[idx]["text"]))\n        source_blocks.append(f"[{n}]\\n" + "\\n".join(meta + passages)[:2200])\n    system = (\n        "Tu es SmartJob Morocco. Réponds en français uniquement à partir des sources. "\n        "N\'invente aucune information. Chaque fait doit avoir une citation [n]. "\n        "Les métadonnées structurées (ville, contrat, expérience, formation) font foi. "\n        "Signale toute contradiction entre métadonnées et texte. "\n        "Si la réponse est absente, dis : Cette information n\'apparaît pas dans les offres retrouvées."\n    )\n    messages = [\n        {"role": "system", "content": system},\n        {"role": "user", "content": "SOURCES :\\n" + "\\n\\n".join(source_blocks) +\n         "\\n\\nQUESTION : " + question + "\\nRéponds brièvement avec les citations."}\n    ]\n    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = tokenizer(prompt, return_tensors="pt")\n    with torch.inference_mode():\n        output = model.generate(\n            **inputs, max_new_tokens=220, do_sample=False,\n            repetition_penalty=1.08, pad_token_id=tokenizer.eos_token_id\n        )\n    answer = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()\n    valid = {int(x) for x in re.findall(r"\\[(\\d+)\\]", answer)}\n    invalid = [x for x in valid if x < 1 or x > len(results)]\n    for x in invalid:\n        answer = answer.replace(f"[{x}]", "")\n    if not re.search(r"\\[\\d+\\]", answer) and "n\'apparaît pas" not in answer.lower():\n        answer += "\\n\\nAucune citation valide détectée : vérifiez la réponse dans les annonces."\n    return answer\n\n\ndef get_filters(query: str, offers: pd.DataFrame, selected_city: str, selected_role: str,\n                selected_contract: str, selected_remote: bool):\n    q = fold(query)\n    filters = {}\n    if selected_city != "Toutes":\n        filters["city"] = selected_city\n    else:\n        cities = sorted(offers["city"].dropna().astype(str).unique(), key=len, reverse=True)\n        for city in cities:\n            if fold(city) in q:\n                filters["city"] = city\n                break\n\n    role_patterns = {\n        "Data Scientist": r"data scientist|machine learning|nlp|computer vision",\n        "Data Engineer": r"data engineer|ingenieur de donnees|big data engineer|data platform",\n        "Data Analyst": r"data analyst|analyste de donnees|analyste data",\n        "BI / Décisionnel": r"business intelligence|decisionnel|bi analyst",\n        "Business Analyst": r"business analyst|analyste metier|analyste fonctionnel",\n    }\n    if selected_role != "Tous":\n        filters["role"] = selected_role\n    else:\n        for role, pattern in role_patterns.items():\n            if re.search(pattern, q):\n                filters["role"] = role\n                break\n\n    if selected_contract != "Tous":\n        filters["contract"] = selected_contract\n\n    tech_patterns = {\n        "Python": r"\\bpython\\b", "SQL": r"\\bsql\\b", "Spark": r"\\bspark\\b|pyspark",\n        "Power BI": r"power\\s*-?bi", "Tableau": r"\\btableau\\b",\n        "Airflow": r"\\bairflow\\b", "Azure": r"\\bazure\\b", "AWS": r"\\baws\\b",\n        "Docker": r"\\bdocker\\b", "Kubernetes": r"kubernetes|\\bk8s\\b",\n        "Hadoop": r"\\bhadoop\\b|hdfs|hive", "Kafka": r"\\bkafka\\b",\n        "Databricks": r"\\bdatabricks\\b", "PostgreSQL": r"postgres",\n        "MongoDB": r"mongodb|\\bmongo\\b",\n    }\n    detected = [name for name, pattern in tech_patterns.items() if re.search(pattern, q)]\n    if detected:\n        filters["techs"] = detected\n        filters["tech_operator"] = "OR" if re.search(r"\\bou\\b", q) else "AND"\n\n    if re.search(r"debutant|debutante|sans experience|premier emploi|junior|stage", q):\n        filters["junior"] = True\n    if selected_remote or re.search(r"teletravail|a distance|remote|hybride", q):\n        filters["remote"] = True\n    return filters\n\n\ndef apply_filters(offers: pd.DataFrame, filters: dict) -> pd.DataFrame:\n    mask = pd.Series(True, index=offers.index)\n    if filters.get("city"):\n        mask &= offers["city"].fillna("").map(fold).eq(fold(filters["city"]))\n    if filters.get("role") and "role" in offers:\n        mask &= offers["role"].fillna("").eq(filters["role"])\n    if filters.get("contract"):\n        mask &= offers["contract"].fillna("").map(fold).eq(fold(filters["contract"]))\n    if filters.get("techs"):\n        op = filters.get("tech_operator", "AND")\n        if op == "OR":\n            mask &= offers["techs"].apply(lambda xs: any(t in (xs or []) for t in filters["techs"]))\n        else:\n            mask &= offers["techs"].apply(lambda xs: all(t in (xs or []) for t in filters["techs"]))\n    if filters.get("junior"):\n        mask &= (\n            offers["experience"].fillna("").str.contains(\n                r"(?i)junior|débutant|debutant|0.?1 an|sans expérience|stage|stagiaire"\n            )\n            | offers["contract"].fillna("").str.contains(r"(?i)stage|stagiaire")\n        )\n    if filters.get("remote"):\n        mask &= offers["remote"].fillna("").str.contains(\n            r"(?i)oui|hybride|remote|télétravail|teletravail"\n        )\n    return offers[mask].copy()\n\n\ndef reciprocal_rank_fusion(rankings, k=60):\n    scores = {}\n    for ranking in rankings:\n        for rank, item in enumerate(ranking, start=1):\n            scores[item] = scores.get(item, 0.0) + 1.0 / (k + rank)\n    return sorted(scores, key=scores.get, reverse=True)\n\n\ndef retrieve(query, offers, chunks, embeddings, bm25_index, embedder, filters, top_k=8,\n             mode="hybride"):\n    candidates = apply_filters(offers, filters) if filters else offers.copy()\n    if candidates.empty:\n        return [], 0\n\n    allowed_ids = set(candidates["offer_id"].astype(str))\n    chunk_offer_ids = chunks["offer_id"].astype(str).tolist()\n    allowed_mask = np.array([oid in allowed_ids for oid in chunk_offer_ids], dtype=bool)\n\n    rankings, chunk_rankings = [], []\n    if mode in ("BM25", "hybride"):\n        scores = np.asarray(bm25_index.get_scores(tokenize(query)), dtype=float)\n        scores[~allowed_mask] = -np.inf\n        valid_chunks = [int(i) for i in np.argsort(-scores)\n                        if np.isfinite(scores[i]) and scores[i] > 0]\n        best = {}\n        for i, score in enumerate(scores):\n            if allowed_mask[i] and np.isfinite(score) and score > 0:\n                oid = chunk_offer_ids[i]\n                best[oid] = max(best.get(oid, -np.inf), float(score))\n        rankings.append(sorted(best, key=best.get, reverse=True))\n        chunk_rankings.append(valid_chunks)\n\n    if mode in ("Dense", "hybride"):\n        qv = embedder.encode(["query: " + query], normalize_embeddings=True,\n                             convert_to_numpy=True).astype("float32")[0]\n        scores = embeddings @ qv\n        scores[~allowed_mask] = -np.inf\n        best = {}\n        for i, score in enumerate(scores):\n            if allowed_mask[i] and np.isfinite(score):\n                oid = chunk_offer_ids[i]\n                best[oid] = max(best.get(oid, -np.inf), float(score))\n        rankings.append(sorted(best, key=best.get, reverse=True))\n        chunk_rankings.append([int(i) for i in np.argsort(-scores) if np.isfinite(scores[i])])\n\n    ranked_offers = rankings[0] if len(rankings) == 1 else reciprocal_rank_fusion(rankings)\n    ranked_chunks = chunk_rankings[0] if len(chunk_rankings) == 1 else reciprocal_rank_fusion(chunk_rankings)\n\n    chunks_by_offer = {}\n    for idx in ranked_chunks:\n        oid = chunk_offer_ids[idx]\n        if oid not in chunks_by_offer:\n            chunks_by_offer[oid] = []\n        if len(chunks_by_offer[oid]) < 3:\n            chunks_by_offer[oid].append(idx)\n\n    offer_lookup = {str(row.offer_id): row for row in candidates.itertuples(index=False)}\n    results = []\n    for oid in ranked_offers:\n        if oid not in offer_lookup:\n            continue\n        results.append((offer_lookup[oid], chunks_by_offer.get(oid, [])))\n        if len(results) >= top_k:\n            break\n    return results, len(candidates)\n\n\ndef render_offer(offer, chunk_ids, chunks, rank):\n    title = str(getattr(offer, "title", "Poste non précisé"))\n    company = getattr(offer, "company", None) or "Entreprise non précisée"\n    city = getattr(offer, "city", None) or "Ville non précisée"\n    contract = getattr(offer, "contract", None) or "Contrat non précisé"\n    experience = getattr(offer, "experience", None) or "Expérience non précisée"\n    techs = getattr(offer, "techs", []) or []\n    url = getattr(offer, "url", None)\n    st.markdown(\'<div class="job-card">\', unsafe_allow_html=True)\n    st.markdown(f"**{rank}. {title}**")\n    st.markdown(f"{company} · {city}")\n    st.markdown(\n        f\'<span class="pill">{contract}</span><span class="pill">{experience}</span>\',\n        unsafe_allow_html=True,\n    )\n    if techs:\n        st.caption("Technologies détectées : " + ", ".join(techs[:8]))\n    excerpts = []\n    for idx in chunk_ids[:2]:\n        if 0 <= idx < len(chunks):\n            text = str(chunks.iloc[idx]["text"])\n            # Retire l\'en-tête répétitif pour rendre l\'extrait plus lisible.\n            excerpts.append(text.split("\\n", 1)[-1] if "\\n" in text else text)\n    if excerpts:\n        st.write(" ".join(excerpts)[:650] + ("…" if len(" ".join(excerpts)) > 650 else ""))\n    else:\n        st.caption("Aucun passage textuel n\'a été associé à cette annonce.")\n    if url and str(url).startswith(("http://", "https://")):\n        st.markdown(f"[Consulter l\'annonce originale ↗]({url})")\n    else:\n        st.caption("Lien original indisponible")\n    st.markdown("</div>", unsafe_allow_html=True)\n\n\ndef analytic_response(question, offers, filters):\n    subset = apply_filters(offers, filters) if filters else offers\n    q = fold(question)\n    if re.search(r"technolog|competence|outil|stack", q):\n        counts = Counter(t for xs in subset["techs"] for t in (xs or []))\n        return "Technologies les plus détectées :\\n\\n" + "\\n".join(\n            f"- **{name}** : {count} annonces" for name, count in counts.most_common(10)\n        )\n    if re.search(r"ville|city", q):\n        counts = subset["city"].fillna("Non précisée").value_counts().head(10)\n        return "Répartition par ville :\\n\\n" + "\\n".join(\n            f"- **{name}** : {count}" for name, count in counts.items()\n        )\n    if re.search(r"entreprise|societe", q):\n        counts = subset["company"].fillna("Non précisée").value_counts().head(10)\n        return "Entreprises les plus représentées :\\n\\n" + "\\n".join(\n            f"- **{name}** : {count}" for name, count in counts.items()\n        )\n    return f"**{len(subset)}** annonce(s) correspondent aux critères détectés, sur **{len(offers)}** annonces."\n\n\n# ---------------------------------------------------------------------\n# UI\n# ---------------------------------------------------------------------\nst.markdown("""\n<div class="main-title">\n  <div class="eyebrow">Marché de l\'emploi · Maroc</div>\n  <h1>SmartJob Morocco</h1>\n</div>\n""", unsafe_allow_html=True)\n\ntry:\n    offers, chunks, embeddings, config, bm25_index = load_data()\nexcept Exception as exc:\n    st.error(str(exc))\n    st.info("Structure attendue : app.py et un dossier smartjob_final_artifacts contenant les trois fichiers Parquet/embeddings et config.json.")\n    st.stop()\n\nwith st.sidebar:\n    st.header("Filtres")\n    cities = ["Toutes"] + sorted(offers["city"].dropna().astype(str).unique().tolist())\n    contracts = ["Tous"] + sorted(offers["contract"].dropna().astype(str).unique().tolist())\n    roles = ["Tous", "Data Scientist", "Data Engineer", "Data Analyst",\n             "BI / Décisionnel", "Business Analyst"]\n    selected_city = st.selectbox("Ville", cities)\n    selected_role = st.selectbox("Métier", roles)\n    selected_contract = st.selectbox("Type de contrat", contracts)\n    selected_remote = st.checkbox("Télétravail / hybride uniquement")\n    mode = st.selectbox("Recherche", ["hybride", "BM25", "Dense"], index=0)\n    top_k = st.slider("Nombre de résultats", min_value=3, max_value=15, value=8)\n    enable_qwen = st.checkbox("Réponse assistée par Qwen", value=False)\n    st.divider()\n    st.caption(f"Source des annonces : {config.get(\'source\', \'ReKrute\')}")\n    st.caption("Prototype de recherche — données issues d\'un instantané de collecte.")\n\nk1, k2, k3 = st.columns(3)\nk1.metric("Offres", f"{len(offers):,}".replace(",", " "))\nk2.metric("Entreprises", f"{offers[\'company\'].nunique():,}".replace(",", " "))\nk3.metric("Villes", f"{offers[\'city\'].nunique():,}".replace(",", " "))\n\nst.markdown(\'<div class="section-label">Recherche d’offres</div>\', unsafe_allow_html=True)\nwith st.form("job_search_form"):\n    query = st.text_input(\n        "Poste ou compétence",\n        placeholder="Ex. Data Analyst, Python, ingénieur commercial…",\n        label_visibility="collapsed",\n    )\n    search_clicked = st.form_submit_button("Rechercher", type="primary", use_container_width=True)\n\nif search_clicked:\n    filters = get_filters(query, offers, selected_city, selected_role,\n                          selected_contract, selected_remote)\n    if not query.strip() and not filters:\n        st.info("Saisissez un poste ou choisissez un filtre.")\n    else:\n        if filters:\n            readable = [f"{k} : {v}" for k, v in filters.items() if k != "tech_operator"]\n            st.caption("Filtres actifs : " + " · ".join(readable))\n        actual_mode = mode\n        embedder = None\n        if mode in ("Dense", "hybride"):\n            try:\n                embedder = load_embedder(config.get(\n                    "embedding_model", "intfloat/multilingual-e5-base"\n                ))\n            except Exception as exc:\n                actual_mode = "BM25"\n                st.warning(\n                    "Recherche sémantique indisponible. Les résultats sont calculés avec BM25. "\n                    "La recherche classique reste disponible."\n                )\n                with st.expander("Détail technique"):\n                    st.code(str(exc))\n        if actual_mode == "Dense" and embedder is None:\n            actual_mode = "BM25"\n\n        if re.search(r"combien|nombre d[\' ]offres|how many", fold(query)):\n            st.info(analytic_response(query, offers, filters))\n        else:\n            with st.spinner("Recherche en cours…"):\n                results, candidate_count = retrieve(\n                    query, offers, chunks, embeddings, bm25_index, embedder,\n                    filters, top_k=top_k, mode=actual_mode\n                )\n            st.caption(\n                f"{candidate_count:,} offres filtrées · {len(results)} résultat(s) · {actual_mode}"\n                .replace(",", " ")\n            )\n            if not results:\n                st.info("Aucune offre correspondante. Élargissez les filtres ou essayez un autre terme.")\n            else:\n                if enable_qwen:\n                    with st.expander("Synthèse des offres avec Qwen"):\n                        st.caption("La génération est exécutée localement et peut être lente sur CPU.")\n                        if st.button("Générer la synthèse"):\n                            try:\n                                tokenizer, qwen_model = load_qwen(\n                                    config.get("llm_model") or "Qwen/Qwen2.5-3B-Instruct"\n                                )\n                                with st.spinner("Génération…"):\n                                    answer = generate_answer(\n                                        query, results, chunks, tokenizer, qwen_model\n                                    )\n                                st.markdown(answer)\n                            except Exception as exc:\n                                st.error(f"Qwen indisponible : {exc}")\n                st.markdown("### Offres correspondantes")\n                for rank, (offer, chunk_ids) in enumerate(results, start=1):\n                    render_offer(offer, chunk_ids, chunks, rank)\n\nwith st.expander("Méthode de recherche"):\n    st.markdown("""\n    **SmartJob Morocco** est un prototype construit à partir d\'un instantané d\'annonces collectées sur ReKrute.\n\n    - **BM25** : recherche lexicale des termes présents dans les annonces.\n    - **Embeddings multilingues** : recherche sémantique avec `multilingual-e5-base`.\n    - **RRF** : fusion des classements lexicaux et sémantiques au niveau des annonces.\n    - **Filtres structurés** : ville, famille de poste, contrat, technologies et télétravail.\n    - **Statistiques** : calculées directement à partir des annonces, sans demander au LLM de compter.\n\n    Les rôles et technologies sont détectés par règles : les résultats peuvent donc comporter des erreurs.\n    Le corpus est un instantané limité aux annonces collectées et ne représente pas tout le marché marocain.\n    """)\n'
REQUIREMENTS_TEXT = '# Versions compatibles avec Python 3.8 sous Windows\nstreamlit==1.39.0\npandas==2.0.3\nnumpy==1.24.4\npyarrow==14.0.2\nrank-bm25==0.2.2\nsentence-transformers==3.0.1\ntransformers==4.41.2\ntokenizers==0.19.1\nhuggingface-hub==0.23.4\nprotobuf==3.20.3\ntorch==2.2.2\n'
README_TEXT = "# SmartJob Morocco — application Streamlit\n\n## 1. Environnement recommandé\n- Windows 10/11\n- Python 3.8 (les versions des dépendances sont épinglées dans `requirements.txt`)\n- Connexion Internet au premier lancement pour télécharger le modèle `intfloat/multilingual-e5-base`.\n\n## 2. Préparer les fichiers\nDécompresser le dossier exporté depuis Kaggle `smartjob_final_artifacts` à côté de `app.py`, de sorte à obtenir :\n\n```text\nsmartjob_streamlit_app_bundle/\n├── app.py\n├── requirements.txt\n└── smartjob_final_artifacts/\n    ├── offers_clean.parquet\n    ├── chunks.parquet\n    ├── embeddings.npy\n    └── config.json\n```\n\n## 3. Installation Windows\nOuvrir PowerShell dans le dossier qui contient `app.py`, puis exécuter :\n\n```powershell\npy -3.8 -m venv .venv\n.\\.venv\\Scripts\\Activate.ps1\npython -m pip install --upgrade pip\npython -m pip install -r requirements.txt\nstreamlit run app.py\n```\n\nSi PowerShell bloque l'activation de l'environnement virtuel, utiliser directement :\n```powershell\n.\\.venv\\Scripts\\python.exe -m pip install -r requirements.txt\n.\\.venv\\Scripts\\python.exe -m streamlit run app.py\n```\n\n## 4. Correction ciblée de l'erreur protobuf\nL'erreur `Descriptors cannot be created directly` est généralement due à une version de `protobuf` trop récente pour une dépendance générée avec une ancienne version de protocole.\n\nDans le même environnement Python que Streamlit :\n```powershell\npython -m pip install --force-reinstall protobuf==3.20.3\n```\nFermer puis relancer Streamlit. Ne pas installer protobuf dans un autre Python global.\n\n## 5. Lancer l'application\n```powershell\nstreamlit run app.py\n```\n\nLa recherche hybride charge le modèle d'embeddings sur CPU. Le premier démarrage télécharge le modèle et peut prendre quelques minutes. Qwen est facultatif et désactivé par défaut ; son exécution locale sur CPU peut être lente et gourmande en mémoire.\n\n## 6. Remarques\n- Cette interface est un prototype basé sur un instantané d'annonces collectées sur ReKrute.\n- Les rôles et technologies détectés automatiquement peuvent comporter des erreurs.\n- Les réponses de Qwen doivent être vérifiées dans les annonces sources.\n- Pour utiliser uniquement BM25, sélectionner `BM25` dans les critères : le modèle d'embeddings ne sera pas chargé.\n"

Path("app.py").write_text(APP_SOURCE, encoding="utf-8")
Path("requirements.txt").write_text(REQUIREMENTS_TEXT, encoding="utf-8")
Path("README_STREAMLIT.md").write_text(README_TEXT, encoding="utf-8")

# N'exporter que les fichiers nécessaires à l'application.
required_artifacts = [
    "offers_clean.parquet",
    "chunks.parquet",
    "embeddings.npy",
    "config.json",
]
missing = [name for name in required_artifacts if not (Path(ARTIFACT_DIR) / name).exists()]
if missing:
    raise FileNotFoundError(
        "Artefacts manquants : " + ", ".join(missing) +
        ". Exécute les cellules de préparation et de sauvegarde avant l'export."
    )

bundle_path = Path("SmartJob_Morocco_App_Finale.zip")
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write("app.py", arcname="app.py")
    zf.write("requirements.txt", arcname="requirements.txt")
    zf.write("README_STREAMLIT.md", arcname="README_STREAMLIT.md")
    for name in required_artifacts + ["relevance_to_label.csv"]:
        artifact = Path(ARTIFACT_DIR) / name
        if artifact.exists():
            zf.write(artifact, arcname=f"smartjob_final_artifacts/{name}")

print("Archive prête :", bundle_path.resolve())
print("Contenu :", zipfile.ZipFile(bundle_path).namelist())


Archive prête : /kaggle/working/SmartJob_Morocco_App_Finale.zip
Contenu : ['app.py', 'requirements.txt', 'README_STREAMLIT.md', 'smartjob_final_artifacts/offers_clean.parquet', 'smartjob_final_artifacts/chunks.parquet', 'smartjob_final_artifacts/embeddings.npy', 'smartjob_final_artifacts/config.json', 'smartjob_final_artifacts/relevance_to_label.csv']
